# Idea 4 · Lesson 4 — Struct2Graph: model + training

We re-implement, in miniature, the architecture Sendin adapted from **Struct2Graph** (Baranwal & Mayank 2022):

1. two chains → two graphs (lesson 3),
2. a **shared-weight GCN** encodes each chain's residues,
3. a **mutual-attention** block couples the two chains' residues,
4. attended context vectors are pooled, concatenated, and classified.

Sendin trained a 31-class PINDER *cluster* classifier (100 PPIs each). PINDER is multi-GB, so as a runnable stand-in we treat a handful of real complexes — all present in SKEMPI v2.0 — as their own classes, with light edge-dropout augmentation for several examples each. Same shape of task; trains in seconds. The full PINDER loader is the natural next step (see the closing notes).

> **Run order matters.** These cells build on each other — run them top to bottom (*Run All*). Heavy steps (training, Integrated Gradients) are small by design but still compute; on a shared machine, run them when the box is idle.

## Setup

In [1]:
import os, sys
HERE = os.path.abspath("")
ROOT = HERE if os.path.isfile(os.path.join(HERE, "ig", "idea4_common.py")) \
    else os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "ig"))
DATA = os.path.join(ROOT, "data")
WEIGHTS = os.path.join(DATA, "idea4_struct2graph.pt")
print("repo root:", ROOT)

repo root: C:\Users\soura\code\2026\plm-starter


In [2]:
from idea4_common import (DEMO_COMPLEXES, build_demo_dataset,
                          Struct2Graph, train_struct2graph,
                          load_or_train_demo, get_device)
import torch
import matplotlib.pyplot as plt
device = get_device(); print('device:', device)

device: cuda


## Step 1 — The miniature multi-class dataset

Each complex below is one class. `build_demo_dataset` downloads each structure, builds the two chain graphs, and adds edge-dropout variants so every class has a few non-identical examples.

In [3]:
for i, (pdb, ca, cb, label) in enumerate(DEMO_COMPLEXES):
    print(f'class {i}: {label:24s} ({pdb} {ca}/{cb})')

class 0: barnase / barstar        (1brs A/D)
class 1: trypsin / BPTI           (2ptc E/I)
class 2: elastase / OMTKY3        (1ppf E/I)
class 3: chymotrypsin / eglin-c   (1acb E/I)
class 4: beta-lactamase / BLIP    (1jtg A/B)


In [4]:
samples = build_demo_dataset(threshold=8.0, augment=6, cache_dir=DATA)
n_classes = len({s['label'] for s in samples})
print(f'{len(samples)} samples across {n_classes} classes')

35 samples across 5 classes


## Step 2 — The model

A compact `Struct2Graph`: 2 GCN layers, hidden width 32, a mutual-attention head, and a small classifier. The forward pass stashes the node embeddings and attention matrix on `model._cache` for the explainability notebooks.

In [5]:
model = Struct2Graph(in_dim=20, hidden=32, num_classes=n_classes)
n_params = sum(p.numel() for p in model.parameters())
print(f'{n_params:,} trainable parameters')

6,021 trainable parameters


Forward pass on one complex, untrained — just to confirm the shapes. The attention matrix is (L_receptor × L_ligand): one weight per pair of residues across the interface.

In [6]:
s0 = next(s for s in samples if not s['augmented'])  # pristine 1brs
logits = model(s0['ga'], s0['gb'])
print('logits shape:', tuple(logits.shape))
print('attention shape (La x Ld):', tuple(model._cache['attn_ab'].shape))

logits shape: (1, 5)
attention shape (La x Ld): (108, 87)


## Step 3 — Train

Categorical cross-entropy, Adam, a few dozen epochs. At this scale the model essentially memorises the classes — which is *on message*: Sendin's headline result was that a 0.97-F1 PINDER model leaned on global shape shortcuts. We are not chasing a number here; we want a trained model whose attention, IG, and GNNExplainer signals the next notebooks can interrogate. We save the weights so lessons 5–7 reuse this exact model.

In [7]:
model = train_struct2graph(samples, hidden=32, epochs=80, lr=5e-3,
                           device=device, verbose=True)
import os
os.makedirs(DATA, exist_ok=True)
torch.save(model.state_dict(), WEIGHTS)
print('saved weights ->', WEIGHTS)

epoch   0  loss 1.6333


epoch  20  loss 0.0002


epoch  40  loss 0.0000


epoch  60  loss 0.0000


epoch  79  loss 0.0000
saved weights -> C:\Users\soura\code\2026\plm-starter\data\idea4_struct2graph.pt


## Step 4 — Training accuracy

On the demo set we expect near-perfect accuracy (it is tiny and the model overfits). The value of the project is **not** this number — it is whether the *explanations* point at real biology, which lesson 7 tests.

In [8]:
model.eval()
correct = 0
with torch.no_grad():
    for s in samples:
        ga, gb = s['ga'].to(device), s['gb'].to(device)
        pred = int(model(ga, gb).argmax(1))
        correct += (pred == s['label'])
print(f'train accuracy: {correct}/{len(samples)} = {correct/len(samples):.2f}')

train accuracy: 35/35 = 1.00


## Closing notes — scaling to real PINDER

To turn this miniature into the actual experiment:

- replace `build_demo_dataset` with a PINDER loader (`pip install pinder`; iterate the 31-cluster holo subset),
- keep the same `Struct2Graph` and training loop,
- optionally swap one-hot node features for ESM-2 embeddings (`gnn_l4` shows how).

Everything downstream (lessons 5–7) is written against the model interface, not the dataset, so it carries over unchanged.